# Custom Task Performance Analysis

Copy of `02_task_performance_analysis`, trimmed to the setup needed for Plot 1 onward.

In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

# Load project

In [ ]:
from pathlib import Path
import pandas as pd
from compass_labyrinth import load_project


project_path = Path(r'D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis')

config, cohort_metadata = load_project(project_path=project_path)
config

# Import Combined DLC CSV

In [ ]:
df_all_csv = pd.read_csv(project_path / "csvs" / "combined" / "preprocessed_combined_file.csv")
df_all_csv

# Create Time-Binned Dictionary

This workflow bins data from each session into defined intervals and calculates normalized region occupancy for each genotype. Each pivot table summarizes the proportion of time spent in each region across all sessions for a given bin.

### Recommended Use:
1. Ensure `df_all_csv` contains 'Session', 'Genotype', and 'Region' columns.
2. Set `LOWER_LIMIT` and `UPPER_LIMIT` to define the full range of timepoints to analyze.
3. Set `BIN_SIZE` (used as `difference`) to determine the duration of each bin (e.g., 10000).
4. Provide `region_lengths` as a dictionary mapping region names to their total grid lengths.
5. The function returns a dictionary of pivot tables (one per genotype), useful for heatmap plotting.

Let's generate pivot tables for each genotype and time bin:

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import generate_region_heatmap_pivots


pivot_dict = generate_region_heatmap_pivots(
    df=df_all_csv,
    lower_lim=0,        # Start of time window
    upper_lim=80000,    # End of time window
    difference=10000,   # Bin width in timepoints
    region_columns=["entry_zone", "reward_path", "overshoot", "target_zone"],
)
pivot_dict

# Exclusion Criteria

This workflow quantifies target zone engagement across bouts and sessions, plots the relationship between target usage and session duration, and interactively excludes low-performing sessions based on user-defined thresholds.

### Recommended Use:
1. Ensure `df_all_csv` contains 'Session', 'Genotype', 'Grid.Number', and 'Bout_Index' columns.
2. Run `compute_frames_per_session()` to get the number of frames per session.
3. Use `compute_target_zone_usage()` to calculate per-bout usage for a specified region; set `region` and `difference` accordingly.
4. Summarize target usage at the session level using `summarize_target_usage()` with `mouseinfo` to add sex/genotype metadata.
5. Visualize the relationship between number of frames and target usage using `plot_target_usage_vs_frames()`.
6. Call `exclude_low_performing_sessions()` to interactively remove sessions below usage and duration thresholds.

Example Workflow:

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    compute_frames_per_session,
    compute_target_zone_usage,
    summarize_target_usage,
    plot_target_usage_vs_frames,
    exclude_low_performing_sessions,
    plot_target_usage_with_exclusions
)


# Set these values
BIN_SIZE = 10000               # BASED ON THE TIME-BINNED DICTIONARY
REGION = "target_zone"         # Region to evaluate usage

# Step 1: Total frames per session
frames_df = compute_frames_per_session(df=df_all_csv)

# Step 2: Target zone usage per bout
region_target = compute_target_zone_usage(
    df=df_all_csv,
    pivot_dict=pivot_dict,
    region=REGION,
    difference=BIN_SIZE,
)

# Step 3: Session-level summary
region_summary = summarize_target_usage(
    region_target=region_target,
    frames_df=frames_df,
    cohort_metadata=cohort_metadata,
)

# Step 4: Visualize usage vs. session duration
plot_target_usage_vs_frames(
    config=config,
    figures_dirname="figures_b/exclusion_criteria",
    summary_df=region_summary,
)

# Step 5: Interactive exclusion based on usage and frame thresholds
df_all_csv = exclude_low_performing_sessions(
    df=df_all_csv,
    summary_df=region_summary,
    usage_threshold=0.4,
    min_frames=30000,
)

# Step 6: Replot the target usage v/s frames plot with the excluded sessions 'X' out
sessions_to_exclude = region_summary.loc[
    ~region_summary['session'].isin(df_all_csv['session'])
]['session'].tolist()

plot_target_usage_with_exclusions(
    config=config,
    figures_dirname="figures_b/exclusion_criteria",
    summary_df=region_summary,
    sessions_to_exclude=sessions_to_exclude,
)

### Subset the Time-Binned Dictionary based on Valid Sessions

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import subset_pivot_dict_sessions


pivot_dict = subset_pivot_dict_sessions(pivot_dict, df_all_csv)
pivot_dict

## Define Custom Zones and Genotypes

`ZONES` defines each zone once, by grid cell number (0-99) -- see `media/new_maze_layout_v2.svg` for the cell layout. `ZONE_SETS` then groups zone names into the combinations used below (a zone can belong to more than one set). Every plot cell just picks a set by name (e.g. `ZONE_NAMES = ZONE_SETS["5_column"]`) instead of redefining cell numbers itself. Add a new zone to `ZONES` or a new combination to `ZONE_SETS` to make it available everywhere.

`GENOTYPE_SETS` is `set_name -> {label: [raw genotype strings]}`. A label mapping to one raw genotype (e.g. `"pair"`) keeps genotypes separate; a label mapping to several (e.g. `"wt_nlf"`'s `"WT": ["E3", "E4"]`) pools them into one group. Both cases are used identically -- there's no separate "raw" vs "pooled" mechanism:

```python
GENOTYPE_SET = "wt_nlf"                                   # or "pair", "all", "apoe_pair", ...
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())       # ["WT", "NLF"]
df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)  # genotype column relabeled/filtered
```

`build_zone_pivot(zone_names, df=df_geno)` then relabels `df_geno` by the chosen zones and builds a matching pivot dict. Sections that are not zone-based (bout success, deviation, backtracking) skip `build_zone_pivot()` and use `df_geno` directly, but still get their genotype scoping from `build_genotype_groups()`.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    generate_region_heatmap_pivots,
    generate_region_heatmap_pivots_by_percentage,
    subset_pivot_dict_sessions,
    get_max_session_row_bracket,
)


LOWER_LIMIT = 0
BIN_SIZE = 5000
UPPER_LIMIT = get_max_session_row_bracket(df_all_csv)

# zone_name -> list of grid cell numbers (media/new_maze_layout_v2.svg). This
# is a mutually exclusive partition of all 100 cells. Organized in three
# groups: sequential maze segments (entry to exit), the 9 individual hairpin
# turns, then pooled/composite zones built from those. Identical to 05b's
# ZONES -- same maze, same cell numbering.
ZONES = {
    # --- Sequential maze segments, entry to exit ---
    "entry_zone": [0, 1],
    "entry_corridor": [2, 3, 4, 5, 6, 7],
    "column_zone": [
        12, 13, 14, 15, 16, 17,
        22, 23, 24, 25, 26, 27,
        32, 33, 34, 35, 36, 37,
        42, 43, 44, 45, 46, 47,
        52, 53, 54, 55, 56, 57,
        62, 63, 64, 65, 66, 67,
    ],
    "column_before_reward": [72, 73, 74, 75, 76, 77],
    "reward_zone": [70, 71, 80, 81],
    "column_after_reward": [82, 83, 84, 85, 86, 87],
    "overshoot_column": [92, 93, 94, 95, 96, 97],
    "terminal": [90, 91],

    # --- The 9 individual turns of the hairpin (T1-T9, alternating bottom/top):
    # T1 (bottom, col0-1), T2 (top, col1-2), T3 (bottom, col2-3), T4 (top,
    # col3-4), T5 (bottom, col4-5), T6 (top, col5-6), T7 (bottom, col6-7),
    # T8 (top, col7-8 -- this is exactly reward_zone above, no separate
    # alias needed), T9 (bottom, col8-9). Used by the "9_corners" ZONE_SET
    # below to give each turn its own line/row instead of the pooled
    # top_corners/bottom_corners groupings.
    "t1_corner": [8, 9, 18, 19],
    "t2_corner": [10, 11, 20, 21],
    "t3_corner": [28, 29, 38, 39],
    "t4_neutral": [30, 31, 40, 41],  # T4
    "t5_corner": [48, 49, 58, 59],
    "t6_corner": [50, 51, 60, 61],
    "t7_corner": [68, 69, 78, 79],
    "t9_corner": [88, 89, 98, 99],

    # --- Pooled/composite zones (groups of the segments/corners above) ---
    "top_corners": [10, 11, 20, 21, 30, 31, 40, 41, 50, 51, 60, 61],  # T2, T4, T6
    "bottom_corners": [28, 29, 38, 39, 48, 49, 58, 59, 68, 69, 78, 79],  # T3, T5, T7
    "all_corners": [
        10, 11, 20, 21, 30, 31, 40, 41, 50, 51, 60, 61,
        28, 29, 38, 39, 48, 49, 58, 59, 68, 69, 78, 79
    ],
    "aft_overshoot_column": [88, 89, 90, 91, 98, 99, 92, 93, 94, 95, 96, 97],
    "overshoot_zone": [82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99],
    "pre-reward_zone": [60,61,62,63,64,65,66,67,68,69,72,73,74,75,76,77,78,79],

    # Original 4 zones from 02_task_performance_analysis.ipynb (matches
    # constants.py's REGION_MAPPING exactly -- entry_zone(2) + reward_path(76)
    # + target_zone(4) + overshoot(18) = 100 cells). "target_zone" is the same
    # 4 cells as "reward_zone" above, just named to match notebook 02.
    "target_zone": [70, 71, 80, 81],
    "reward_path": (
        [2, 3, 4, 5, 6, 7]  # entry_corridor
        + [12, 13, 14, 15, 16, 17, 22, 23, 24, 25, 26, 27, 32, 33, 34, 35, 36, 37,
           42, 43, 44, 45, 46, 47, 52, 53, 54, 55, 56, 57, 62, 63, 64, 65, 66, 67]  # column_zone
        + [72, 73, 74, 75, 76, 77]  # column_before_reward
        + [8, 9, 18, 19]  # t1_corner
        + [10, 11, 20, 21, 30, 31, 40, 41, 50, 51, 60, 61]  # top_corners
        + [28, 29, 38, 39, 48, 49, 58, 59, 68, 69, 78, 79]  # bottom_corners
    ),
    "overshoot": (
        [82, 83, 84, 85, 86, 87]  # column_after_reward
        + [92, 93, 94, 95, 96, 97]  # overshoot_column
        + [88, 89, 90, 91, 98, 99]  # t9_corner + terminal
    ),
}

# Group zone names into the combinations used by the plots below (a zone can be in more than one set)
ZONE_SETS = {
    "5_columns": ["entry_corridor", "column_zone", "column_before_reward", "column_after_reward", "overshoot_column"],
    "4_corners": ["top_corners", "bottom_corners", "t4_neutral", "reward_zone"],
    "six_zones": ["all_corners", "column_zone", "column_before_reward", "reward_zone", "column_after_reward", "aft_overshoot_column"],
    "reward_zones": ["column_before_reward", "reward_zone", "column_after_reward"],
    "entry_reward": ["entry_zone", "entry_corridor", "top_corners", "bottom_corners", "column_zone", "reward_zone"],
    "original_4": ["entry_zone", "reward_path", "target_zone", "overshoot"],
    "except_entry": [
        "entry_corridor", "column_zone", "column_before_reward", "column_after_reward", "overshoot_column",
        "t1_corner", "top_corners", "bottom_corners", "reward_zone", "t9_corner", "terminal"
        ],
    "9_corners": [
        "t1_corner", "t2_corner", "t3_corner", "t4_neutral", "t5_corner",
        "t6_corner", "t7_corner", "reward_zone", "t9_corner",
    ],
    "around_reward_zones": ["pre-reward_zone", "reward_zone", "overshoot_zone"],
}

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, 9_corners, wt_nlf, apoe_pair

# set_name -> {label: [raw genotype strings that map to that label]}. A
# label mapping to a single raw genotype (e.g. "pair") keeps genotypes
# separate; a label mapping to several (e.g. "wt_nlf"'s "WT": ["E3", "E4"])
# pools them into one group. Both cases are used identically everywhere via
# build_genotype_groups() -- there's no separate "raw" vs "pooled" mechanism.
GENOTYPE_SETS = {
    "pair": {"E3": ["E3"], "E4": ["E4"]},
    "all": {"E3": ["E3"], "E4": ["E4"], "E3NLF": ["E3NLF"], "E4NLF": ["E4NLF"]},
    "apoe3_group": {"E3": ["E3"], "E3NLF": ["E3NLF"]},  # E3 vs E3NLF -- NLF's effect within the APOE3 background
    "apoe4_group": {"E4": ["E4"], "E4NLF": ["E4NLF"]},  # E4 vs E4NLF -- NLF's effect within the APOE4 background
    "apoe_pair": {"APOE3": ["E3", "E3NLF"], "APOE4": ["E4", "E4NLF"]},  # pooled by APOE allele
    "wt_nlf": {"WT": ["E3", "E4"], "NLF": ["E3NLF", "E4NLF"]},  # pooled by NLF modifier
}


def build_zone_pivot(zone_names, df=None):
    """Relabel a dataframe by the given zones (looked up in ZONES) and build a matching pivot_dict.

    zone_names : list of zone names (keys of ZONES) to include -- zones may
        overlap (share grid cells); a shared cell's frames are counted
        toward every zone it belongs to (not just one), by building each
        zone's rows independently instead of a single cell->zone lookup.
    df : dataframe to build from (defaults to df_all_csv). Pass the output of
        build_genotype_groups() to scope to a particular GENOTYPE_SETS entry.
    """
    if df is None:
        df = df_all_csv
    zones = {name: ZONES[name] for name in zone_names}
    labeled = []
    for zone, cells in zones.items():
        subset = df[df["grid_number"].isin(cells)].copy()
        subset["region"] = zone
        labeled.append(subset)
    # Concat WITHOUT ignore_index, then sort_index to restore each session's
    # original chronological row order -- generate_region_heatmap_pivots() bins
    # by row POSITION (not by timestamp), so losing that order (as ignore_index=True
    # did) made each "bin" land almost entirely inside one zone's block instead of
    # a real time window, producing near-0/near-1 density artifacts.
    df_zones = (
        pd.concat(labeled).sort_index(kind="stable").reset_index(drop=True)
        if labeled else df.iloc[0:0].assign(region=[])
    )

    pivot_dict_zones = generate_region_heatmap_pivots(
        df=df_zones,
        lower_lim=LOWER_LIMIT,
        upper_lim=UPPER_LIMIT,
        difference=BIN_SIZE,
        region_columns=zone_names,
        region_lengths={zone: len(cells) for zone, cells in zones.items()},
    )
    pivot_dict_zones = subset_pivot_dict_sessions(pivot_dict_zones, df_zones)
    return df_zones, pivot_dict_zones


def build_zone_pivot_by_percentage(zone_names, df=None, n_bins=5):
    """Like build_zone_pivot(), but bins each session by percentage of its own
    total frame count (first 20%, next 20%, ... for n_bins=5) instead of a
    fixed absolute frame range -- every session contributes to every bin,
    comparing the same fraction of session progress instead of the same raw
    frame count (which cuts short sessions out of later bins entirely).

    zone_names : list of zone names (keys of ZONES) to include
    df : RAW (not zone-filtered) dataframe to build from, defaults to
        df_all_csv -- unlike build_zone_pivot(), no pre-filtering to zone
        cells is needed since region membership is computed internally per
        percentage segment (each session's own total row count is required
        to compute the percentage bin edges).
    n_bins : number of equal-percentage segments per session (default 5 -> 20% each)
    """
    if df is None:
        df = df_all_csv
    zones = {name: ZONES[name] for name in zone_names}
    return generate_region_heatmap_pivots_by_percentage(
        df=df,
        zones=zones,
        region_columns=zone_names,
        n_bins=n_bins,
    )


def build_genotype_groups(df, set_name):
    """Relabel a dataframe's `genotype` column per GENOTYPE_SETS[set_name].

    Rows whose raw genotype isn't in any label's list are dropped; rows that
    are get `genotype` replaced by the label (e.g. E3 and E3NLF both become
    "APOE3" for set_name="apoe_pair"). Works the same whether the set keeps
    genotypes separate ("pair") or pools several into one label ("wt_nlf").

    set_name : key of GENOTYPE_SETS to use
    """
    mapping = GENOTYPE_SETS[set_name]
    genotype_lookup = {raw: label for label, raw_genotypes in mapping.items() for raw in raw_genotypes}
    df_grouped = df[df["genotype"].isin(genotype_lookup)].copy()
    df_grouped["genotype"] = df_grouped["genotype"].map(genotype_lookup)
    return df_grouped


# Plot 1: Heatmap Representations

Use this function to visualize how region occupancy evolves over time for a specific genotype or group in your behavioral dataset. Each heatmap corresponds to a fixed-length time bin and shows normalized region occupancy per session, allowing for temporal comparison.

**How the values are normalized:** for each zone, raw frame counts are first divided by that zone's cell count (so a 36-cell zone isn't favored over a 6-cell zone just for being bigger) -- this gives a density (frames per cell). Those densities are then rescaled so they sum to 100%, but **only across the zones in `ZONE_NAMES`** for that plot, not the whole maze. Any time spent in a zone you didn't include (e.g. the reward zone, if you're using `"5_column"`) doesn't show up as a gap -- it's silently redistributed across whichever zones you did include. This means the same zone's percentage isn't directly comparable between plots that use a different `ZONE_NAMES` set.

### Recommended Use:
1. Ensure that `pivot_dict` is generated using `generate_region_heatmap_pivots()`.
2. Specify the genotype/group you want to plot (e.g., 'WT').
3. Define the time binning parameters: `lower_lim`, `upper_lim`, and `difference`.
4. Optionally set a custom region order and vmax for consistent color scaling.
5. Run the function to display vertically stacked heatmaps across time bins.

Example Workflow:

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_region_heatmaps


ZONE_SET = "5_column"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "pair"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
VMAX = 1.0  # max range on colorbar
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"  # e.g. region_heatmaps_E3_pair_5_column.pdf

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
df_zones, pivot_dict_zones = build_zone_pivot(ZONE_NAMES, df=df_geno)

for genotype in GENOTYPES:
    valid_sessions = df_zones[df_zones.genotype == genotype]['session'].unique().tolist()
    if len(valid_sessions) == 0:
        print(f"No valid sessions for genotype '{genotype}', skipping.")
        continue

    plot_region_heatmaps(
        config=config,
        figures_dirname="figures_b/plot1_heatmap",
        pivot_dict=pivot_dict_zones,
        group_name=genotype,
        lower_lim=LOWER_LIMIT,
        upper_lim=UPPER_LIMIT,
        difference=BIN_SIZE,
        vmax=VMAX,
        included_sessions=valid_sessions,
        region_desired_order=ZONE_NAMES,
        filename_suffix=FILENAME_SUFFIX,
    )


### EXTRA: Heatmap Representations across all Genotypes (when multiple genotypes)

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_region_heatmaps_all_genotypes

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, wt_nlf, apoe_pair
ZONE_SET = "except_entry"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "wt_nlf"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
VMAX = 0.7  # max range on colorbar -- real data for original_4/wt_nlf reaches 1.0 (single-zone bins)
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
df_zones, pivot_dict_zones = build_zone_pivot(ZONE_NAMES, df=df_geno)

plot_region_heatmaps_all_genotypes(
    config=config,
    figures_dirname="figures_b/plot1_heatmap_all_genotypes",
    pivot_dict=pivot_dict_zones,
    df_all_csv=df_zones,
    lower_lim=LOWER_LIMIT,
    upper_lim=UPPER_LIMIT,
    difference=BIN_SIZE,
    included_genotypes=GENOTYPES,
    region_desired_order=ZONE_NAMES,
    spacing_w=0.2,
    spacing_h=0.15,
    show_colorbar=True,
    vmax=VMAX,
    filename_suffix=FILENAME_SUFFIX,
)


# Plot 1b: Heatmap Representations by Percentage of Session

Same idea as Plot 1, but bins each session by percentage of its own total frame count (first 20%, next 20%, ... for `N_BINS=5`) instead of a fixed absolute frame range. Session lengths vary a lot (see the earlier `UPPER_LIMIT` discussion) -- fixed-frame bins mean a short session runs out of rows and drops out of later bins entirely, so most of a tall heatmap ends up representing just 1-2 long outlier sessions. Percentage bins instead let every session contribute to every bin, comparing "first 20% of the maze" for one mouse against "first 20%" for another, regardless of how many raw frames that took.

### Recommended Use:
1. Pick a `ZONE_SET`/`GENOTYPE_SET` and `N_BINS` (default 5 -> 20% segments).
2. Use `build_zone_pivot_by_percentage()` -- pass the RAW (not zone-filtered) genotype dataframe; it computes each session's own row count internally.
3. Plot with `plot_region_heatmaps()`/`plot_region_heatmaps_all_genotypes()`, same as Plot 1, but with `lower_lim=0, upper_lim=100, difference=100/N_BINS, bin_label_suffix="%"` so bin titles read "0%-20%" etc.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_region_heatmaps


# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, wt_nlf, apoe_pair
ZONE_SET = "5_column"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "pair"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
N_BINS = 5  # -> 20% segments
VMAX = 1.0  # max range on colorbar
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}_pct"

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
pivot_dict_zones = build_zone_pivot_by_percentage(ZONE_NAMES, df=df_geno, n_bins=N_BINS)

for genotype in GENOTYPES:
    valid_sessions = df_geno[df_geno.genotype == genotype]['session'].unique().tolist()
    if len(valid_sessions) == 0:
        print(f"No valid sessions for genotype '{genotype}', skipping.")
        continue

    plot_region_heatmaps(
        config=config,
        figures_dirname="figures_b/plot1b_heatmap_pct",
        pivot_dict=pivot_dict_zones,
        group_name=genotype,
        lower_lim=0,
        upper_lim=100,
        difference=100 // N_BINS,
        bin_label_suffix="%",
        vmax=VMAX,
        included_sessions=valid_sessions,
        region_desired_order=ZONE_NAMES,
        filename_suffix=FILENAME_SUFFIX,
    )

### EXTRA: Heatmap Representations across all Genotypes (when multiple genotypes)

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_region_heatmaps_all_genotypes


# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, wt_nlf, apoe_pair
ZONE_SET = "except_entry"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "wt_nlf"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
N_BINS = 5  # -> 20% segments
VMAX = 0.7  # max range on colorbar -- real data for original_4/wt_nlf reaches 1.0 (single-zone bins)
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}_pct"

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
pivot_dict_zones = build_zone_pivot_by_percentage(ZONE_NAMES, df=df_geno, n_bins=N_BINS)

plot_region_heatmaps_all_genotypes(
    config=config,
    figures_dirname="figures_b/plot1b_heatmap_pct_all_genotypes",
    pivot_dict=pivot_dict_zones,
    df_all_csv=df_geno,
    lower_lim=0,
    upper_lim=100,
    difference=100 // N_BINS,
    bin_label_suffix="%",
    included_genotypes=GENOTYPES,
    region_desired_order=ZONE_NAMES,
    spacing_w=0.2,
    spacing_h=0.15,
    show_colorbar=True,
    vmax=VMAX,
    filename_suffix=FILENAME_SUFFIX,
)

# Shannon's Entropy

This function calculates Shannon entropy over time, reflecting the diversity of region occupancy for each mouse/session within each time bin. A higher entropy indicates more distributed (less selective) region occupancy, whereas lower entropy suggests focused behavior within fewer regions.

### Recommended Use:
1. Ensure `pivot_dict` has been generated using `generate_region_heatmap_pivots()`.
2. Ensure `cohort_metadata` includes metadata 

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import compute_shannon_entropy_per_bin


ZONE_SET = "5_column"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "all"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
df_zones, pivot_dict_zones = build_zone_pivot(ZONE_NAMES, df=df_geno)

# Compute entropy values -- higher = occupancy spread evenly across zones,
# lower = concentrated in fewer zones (a preference signal)
entropy_df = compute_shannon_entropy_per_bin(
    pivot_dict=pivot_dict_zones,
    df_all_csv=df_zones,
    bin_size=BIN_SIZE,
)
entropy_df


## Plot 2: Plotting Shannon's Entropy across Sessions (/Mice)

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_entropy_over_bins


# Plot entropy over time
plot_entropy_over_bins(
    config=config,
    figures_dirname="figures_b/plot2_entropy",
    entropy_df=entropy_df,
    filename_suffix=FILENAME_SUFFIX,
)

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_entropy_over_bins_zoomed


# Zoomed-in view, cut off where sample size gets thin -- bin 40000 is the last
# bin with every genotype still represented; annotated below the x-axis
plot_entropy_over_bins_zoomed(
    config=config,
    figures_dirname="figures_b/plot2_entropy",
    entropy_df=entropy_df,
    cutoff_bin=40000,
    df_all_csv=df_all_csv,
    filename_suffix=FILENAME_SUFFIX,
)

### Statistical tests - Entropy

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    run_entropy_anova,
    run_fdr_pairwise_tests,
    run_mixed_model_per_genotype_pair,
)

## Need more than one genotype to run these

# Repeated Measures ANOVA
anova_result = run_entropy_anova(entropy_df)

# Pairwise t-tests + FDR correction (per bin, per genotype pair)
fdr_results = run_fdr_pairwise_tests(entropy_df)
print(fdr_results)

# Run per-pair mixed models
mixed_results, interaction_table = run_mixed_model_per_genotype_pair(entropy_df)

# View pairwise Bin × Genotype interaction summary
print(interaction_table)

# Proportion of Region-based usage across Time bins

This function computes the proportion of usage of a specified region across sequential time bins, grouped by session and genotype, using the binned pivot tables.

### Recommended Use:
1. Ensure `pivot_dict` is the output from `generate_region_heatmap_pivots()` and contains binned region usage per genotype.
2. Use `cohort_metadata` to map session IDs to genotypes for grouping.
3. Set `region` to the name of the region you want to evaluate (e.g., "target_zone").
4. Set `bin_size` to match the time resolution used when generating `pivot_dict`.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import compute_region_usage_over_bins

# 5_column, 4_corner, six_zones, reward_zones, entry_reward, wt_nlf, apoe_pair
ZONE_SET = "5_column"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "pair"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
df_zones, pivot_dict_zones = build_zone_pivot(ZONE_NAMES, df=df_geno)

# Compute usage over time bins for each zone
reg_binned_by_zone = {
    zone: compute_region_usage_over_bins(
        pivot_dict=pivot_dict_zones,
        df_all_csv=df_zones,
        region=zone,
        bin_size=BIN_SIZE,
    )
    for zone in ZONE_NAMES
}


## Plot 3: Proportion of usage per Region across time

Plot the region usage over time bins:

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_region_usage_over_bins


for zone, reg_binned in reg_binned_by_zone.items():
    plot_region_usage_over_bins(
        config=config,
        figures_dirname="figures_b/plot3_region_usage",
        region_data=reg_binned,
        region_name=zone,
        filename_suffix=FILENAME_SUFFIX,
    )


### Statistical tests - region usage

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    run_region_usage_stats_mixedlm,
    run_region_usage_stats_fdr,
)


# Run a Mixed Effects Model
run_region_usage_stats_mixedlm(reg_binned, region_col=REGION)

# Run pairwise comparison across genotypes with FDR correction
fdr_results = run_region_usage_stats_fdr(reg_binned, region_col=REGION)

## Plot 4: Proportion of usage across all Regions

Plot usage over time bins for all Regions:

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_all_regions_usage_over_bins

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, wt_nlf, apoe_pair
ZONE_SET = "except_entry"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

YLIM = (0, 0.4)      # max proportion shown on the y-axis
MAX_FRAMES = 50000  # cap the x-axis at this frame count (None = show every bin)

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
df_zones, pivot_dict_zones = build_zone_pivot(ZONE_NAMES, df=df_geno)

plot_all_regions_usage_over_bins(
    config=config,
    figures_dirname="figures_b/plot4_all_regions_usage",
    pivot_dict=pivot_dict_zones,
    df_all_csv=df_zones,
    region_list=ZONE_NAMES,
    bin_size=BIN_SIZE,
    ylim=YLIM,
    max_frames=MAX_FRAMES,
    filename_suffix=FILENAME_SUFFIX,
)


## Plot 4b: Proportion of usage across all Regions, by Percentage of Session

Same as Plot 4, but built from a percentage-based `pivot_dict` (`build_zone_pivot_by_percentage()`) instead of a fixed-frame one -- see Plot 1b for why: every session contributes to every bin (first 20%, next 20%, ...) instead of short sessions dropping out of later bins.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_all_regions_usage_over_bins

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, wt_nlf, apoe_pair
ZONE_SET = "except_entry"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "wt_nlf"
N_BINS = 10  # -> 10% segments
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}_pct"

YLIM = (0, 0.4)  # max proportion shown on the y-axis

df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)
pivot_dict_zones = build_zone_pivot_by_percentage(ZONE_NAMES, df=df_geno, n_bins=N_BINS)

plot_all_regions_usage_over_bins(
    config=config,
    figures_dirname="figures_b/plot4b_all_regions_usage_pct",
    pivot_dict=pivot_dict_zones,
    df_all_csv=df_geno,
    region_list=ZONE_NAMES,
    bin_size=100 // N_BINS,
    ylim=YLIM,
    xlabel="% of Session",
    filename_suffix=FILENAME_SUFFIX,
)

# Bout-Level Success Metrics

## Plot 5: Cumulative Successful Bout Percentage

A bout is defined as a continuous sequence starting from the Entry Zone, passing through non-entry nodes, and returning to the Entry Zone. 

A successful bout is defined as one in which the animal reaches the Target Zone at least once before returning to the Entry Zone.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    assign_bout_indices_from_entry_zone,
    summarize_bout_success_by_session,
    plot_success_rate,
)
from compass_labyrinth.constants import ENTRY_ZONE


GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}"
df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)

# 1. Assign bout indices to the navigation dataframe
# entry_zone is now [0, 1] (grown from just [0] -- see constants.py), so
# delimiter_nodes=ENTRY_ZONE catches sessions that never happen to have
# tracked data in cell 0 exactly but do in its neighbor cell 1
df_all_csv_wbouts = assign_bout_indices_from_entry_zone(df_geno, delimiter_nodes=ENTRY_ZONE)

# 2. Compute success summary statistics
bout_summary = summarize_bout_success_by_session(df_all_csv_wbouts)

# 3. Plot % of successful bouts per genotype
plot_success_rate(
    config=config,
    figures_dirname="figures_b/plot5_success_rate",
    summary_table=bout_summary,
    filename_suffix=FILENAME_SUFFIX,
)


### Statistical tests - genotypes

Perform t-test on Success Rate:

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import perform_genotype_ttests


ttest_results_success = perform_genotype_ttests(bout_summary, rate_col='success_rate')
print("T-Test on Success Rate:")
for k, v in ttest_results_success.items():
    print(f"{k}: t={v['t_stat']:.3f}, p={v['p_value']:.4f}, mean1={v['mean_1']:.2f}, mean2={v['mean_2']:.2f}")

## Plot 6: Time-based Successful Bout Percentage

This function computes the percentage of successful bouts within sequential time bins across all sessions, allowing time-resolved assessment of goal-directed behavior.

### Recommended Use:
1. Ensure `df_all_csv` includes 'Session', 'Grid.Number', 'Bout_Index', and time-ordered rows.
2. Set `lower_succ_lim` and `upper_succ_lim` to define the full range of timepoints to analyze.
3. Set `diff_succ` as the size of each time bin (e.g., 5000).
4. Use `valid_bout_threshold` to exclude short bouts (e.g., < 19 steps).
5. Set `optimal_path_regions` to define which regions are required for optimal navigation.
6. Set `target_zone` to specify the region that marks a successful bout.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    assign_bout_indices_from_entry_zone,
    compute_binned_success_summary,
)
from compass_labyrinth.constants import ENTRY_ZONE


GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}"
df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)

# Set these values
BIN_SIZE = 5000
LOWER_BOUND = 0
UPPER_BOUND = 90000

df_all_csv_wbouts_plot6 = assign_bout_indices_from_entry_zone(df_geno, delimiter_nodes=ENTRY_ZONE)

summary_df = compute_binned_success_summary(
    df_all_csv=df_all_csv_wbouts_plot6,
    lower_succ_lim=LOWER_BOUND,
    upper_succ_lim=UPPER_BOUND,
    diff_succ=BIN_SIZE,
    valid_bout_threshold=19,
    optimal_path_regions=['entry_zone', 'reward_path', 'target_zone'],
    target_zone='target_zone'
)
summary_df


In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_binned_success


# Visualize the percentage of successful bouts within each time bin
plot_binned_success(
    config=config,
    figures_dirname="figures_b/plot6_time_based_success",
    summary_df=summary_df,
    filename_suffix=FILENAME_SUFFIX,
)

## Plot 6b: Time-based Successful Bout Percentage, by Percentage of Session

Same as Plot 6, but bins each session by percentage of its own total frame count (first 20%, next 20%, ...) instead of a fixed absolute frame range -- same idea as Plot 1b/4b: every session contributes to every bin, comparing "first 20% of the session" for one mouse against "first 20%" for another, regardless of how many raw frames that took.

### Recommended Use:
1. Ensure `df_all_csv` includes 'Session', 'Grid.Number', 'Bout_Index', and time-ordered rows.
2. Set `N_BINS` (default 5 -> 20% segments).
3. Use `valid_bout_threshold` to exclude short bouts (e.g., < 19 steps).
4. Set `optimal_path_regions` to define which regions are required for optimal navigation.
5. Set `target_zone` to specify the region that marks a successful bout.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    assign_bout_indices_from_entry_zone,
    compute_binned_success_summary_by_percentage,
)
from compass_labyrinth.constants import ENTRY_ZONE


GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_pct"
df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)

# Set these values
N_BINS = 10  # -> 10% segments

df_all_csv_wbouts_plot6b = assign_bout_indices_from_entry_zone(df_geno, delimiter_nodes=ENTRY_ZONE)

summary_df_pct = compute_binned_success_summary_by_percentage(
    df_all_csv=df_all_csv_wbouts_plot6b,
    n_bins=N_BINS,
    valid_bout_threshold=19,
    optimal_path_regions=['entry_zone', 'reward_path', 'target_zone'],
    target_zone='target_zone'
)
summary_df_pct

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_binned_success


# Visualize the percentage of successful bouts within each percentage-of-session bin
plot_binned_success(
    config=config,
    figures_dirname="figures_b/plot6b_time_based_success_pct",
    summary_df=summary_df_pct,
    xlabel="% of Session",
    filename_suffix=FILENAME_SUFFIX,
)

### Statistical tests - time-based

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    run_mixedlm_with_nans,
    run_repeated_measures_anova,
    run_pairwise_comparisons,
)


# Run Mixed Linear Model (preserves NaNs)
#run_mixedlm_with_nans(summary_df)

# Run Repeated Measures ANOVA (NaNs filled with 0)
run_repeated_measures_anova(summary_df)

# Run Pairwise Comparisons (with FDR correction)
run_pairwise_comparisons(summary_df)

# Plot 7: Zone Occupancy and Velocity

This function computes, per bout, the occupancy fraction of each zone in a chosen `ZONE_SET` (instead of a single overshoot/deviation fraction), normalizes and smooths each zone's occupancy plus velocity, and fits exponential curves to the processed data -- one line per zone. Pass `zone_names=["overshoot"]` to reproduce the original single-line "deviation" view exactly.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONE_SET` from `ZONE_SETS` (any number of zones -- one line per zone).
2. Use `compute_zone_occupancy_velocity()` + `process_zone_occupancy_velocity()` to get per-genotype fits.
3. Use `plot_zone_occupancy_velocity_fit()` to visualize the trajectory of zone occupancy per genotype.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    ensure_velocity_column,
    ensure_bout_indices,
    compute_zone_occupancy_velocity,
    process_zone_occupancy_velocity,
)
from compass_labyrinth.constants import ENTRY_ZONE


# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, wt_nlf, apoe_pair
GENOTYPE_SET = "wt_nlf"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
ZONE_SET = "around_reward_zones"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
N_BINS = 400  # resolution for binning bout number before averaging/smoothing
# Y-axis limits for every Plot 7 figure below (this cell's downstream cells) --
# real zone-occupancy fractions are often well under 1.0, so narrow this
# (e.g. (0, 0.1)) to use the plot area better and see more line/CI detail.
OCCUPANCY_YLIM = (0, 0.5)
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"
df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)

# Ensures velocity column exists
df_all_csv_wvelocity = ensure_velocity_column(
    df=df_geno,
    frame_rate=5.0,
)

# Ensure Bout Index column Bout_ID exists
df_all_csv_wbouts = ensure_bout_indices(
    df=df_all_csv_wvelocity,
    delimiter_node=ENTRY_ZONE,
)

# Compute per-zone occupancy and velocity per bout -- one line per zone in
# ZONE_NAMES instead of a single overshoot/deviation fraction
df_occupancy = compute_zone_occupancy_velocity(df_all_csv_wbouts, zones=ZONES, zone_names=ZONE_NAMES)

# Process occupancy and velocity (normalize, smooth, fit curves) per genotype
occupancy_fits = {
    genotype: process_zone_occupancy_velocity(index_df=df_occupancy, genotype=genotype, zone_names=ZONE_NAMES, n_bins=N_BINS)
    for genotype in GENOTYPES
}

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_zone_occupancy_velocity_fit


# Plots smoothed per-zone occupancy trajectories across bouts, overlaid with exponential fit curves
for genotype, (df_zones_p, df_bouts_p, smooth_by_zone, vel_smooth, params_by_zone, params_vel) in occupancy_fits.items():
    plot_zone_occupancy_velocity_fit(
        config=config,
        figures_dirname="figures_b/plot7_zone_occupancy_velocity",
        smooth_by_zone=smooth_by_zone,
        vel_smooth=vel_smooth,
        params_by_zone=params_by_zone,
        params_vel=params_vel,
        genotype=genotype,
        zone_names=ZONE_NAMES,
        max_bouts=200,
        show_velocity=False,
        ylim=OCCUPANCY_YLIM,
        filename_suffix=FILENAME_SUFFIX,
    )

### Zone Occupancy and Velocity for all Genotypes

Plot Zone Occupancy-Velocity plots for all genotypes

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_zone_occupancy_velocity_all


GENOTYPE_SET = "wt_nlf"

plot_zone_occupancy_velocity_all(
    config=config,
    figures_dirname="figures_b/plot7_zone_occupancy_velocity",
    index_df=df_occupancy,
    zone_names=ZONE_NAMES,
    max_bouts=200,
    n_bins=N_BINS,
    show_velocity=False,
    ylim=OCCUPANCY_YLIM,
    filename_suffix=f"_{GENOTYPE_SET}_{ZONE_SET}",
)

### Zone Occupancy and Velocity per Genotype (Successful Bouts Only)

Unsuccessful bouts (never reaching target_zone) can't reach overshoot either -- they mechanically contribute occupancy=0 for overshoot-adjacent zones, which dilutes the real signal rather than reflecting it. Restricting to bouts that reached target_zone at least once isolates the genuine behavior.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    compute_zone_occupancy_velocity,
    plot_zone_occupancy_velocity_all,
)


GENOTYPE_SET = "wt_nlf"

# Restrict to bouts that actually reached the target zone
df_occupancy_successful = compute_zone_occupancy_velocity(
    df_all_csv_wbouts,
    zones=ZONES,
    zone_names=ZONE_NAMES,
    successful_only=True,
)

plot_zone_occupancy_velocity_all(
    config=config,
    figures_dirname="figures_b/plot7_zone_occupancy_velocity",
    index_df=df_occupancy_successful,
    zone_names=ZONE_NAMES,
    max_bouts=200,
    n_bins=N_BINS,
    show_velocity=False,
    ylim=OCCUPANCY_YLIM,
    filename_suffix=f"_{GENOTYPE_SET}_{ZONE_SET}_successful_only",
)

# Plot 7b: Zone Occupancy and Velocity, by Percentage of Bouts

Same as Plot 7 (per-zone occupancy fractions + velocity), but each bout's position is expressed as a percentage of that session's own total bout count (0-100%) instead of a raw sequential bout number. Sessions vary a lot in how many bouts a mouse attempts; comparing "bout 50" across a mouse with 60 total bouts and one with 300 total bouts means very different things, while "80% through this mouse's own bouts" is directly comparable across mice.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONE_SET` from `ZONE_SETS` (any number of zones -- one line per zone).
2. Use `compute_zone_occupancy_velocity_by_percentage()` + `process_zone_occupancy_velocity()` to get per-genotype fits.
3. Use `plot_zone_occupancy_velocity_fit()` to visualize the trajectory of zone occupancy per genotype.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    ensure_velocity_column,
    ensure_bout_indices,
    compute_zone_occupancy_velocity_by_percentage,
    process_zone_occupancy_velocity,
    plot_zone_occupancy_velocity_fit,
)
from compass_labyrinth.constants import ENTRY_ZONE


# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, wt_nlf, apoe_pair
GENOTYPE_SET = "wt_nlf"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
ZONE_SET = "reward_zones"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
N_BINS = 400  # resolution for binning % of bouts before averaging/smoothing
# Y-axis limits for every Plot 7b figure below (this cell's downstream cells) --
# real zone-occupancy fractions are often well under 1.0, so narrow this
# (e.g. (0, 0.1)) to use the plot area better and see more line/CI detail.
OCCUPANCY_YLIM_PCT = (0, 1)
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}_pct"
df_geno = build_genotype_groups(df_all_csv, GENOTYPE_SET)

# Ensures velocity column exists
df_all_csv_wvelocity = ensure_velocity_column(
    df=df_geno,
    frame_rate=5.0,
)

# Ensure Bout Index column Bout_ID exists
df_all_csv_wbouts = ensure_bout_indices(
    df=df_all_csv_wvelocity,
    delimiter_node=ENTRY_ZONE,
)

# Compute per-zone occupancy and velocity per bout, indexed by % of that
# session's own total bout count (0-100%) instead of a raw bout number
df_occupancy_pct = compute_zone_occupancy_velocity_by_percentage(df_all_csv_wbouts, zones=ZONES, zone_names=ZONE_NAMES)

# Process (normalize, smooth, fit curves) per genotype
occupancy_fits_pct = {
    genotype: process_zone_occupancy_velocity(index_df=df_occupancy_pct, genotype=genotype, zone_names=ZONE_NAMES, n_bins=N_BINS)
    for genotype in GENOTYPES
}

for genotype, (df_zones_p, df_bouts_p, smooth_by_zone, vel_smooth, params_by_zone, params_vel) in occupancy_fits_pct.items():
    plot_zone_occupancy_velocity_fit(
        config=config,
        figures_dirname="figures_b/plot7b_zone_occupancy_velocity_pct",
        smooth_by_zone=smooth_by_zone,
        vel_smooth=vel_smooth,
        params_by_zone=params_by_zone,
        params_vel=params_vel,
        genotype=genotype,
        zone_names=ZONE_NAMES,
        max_bouts=100,
        xlabel="% of Bouts",
        title_suffix=" (% of Bouts)",
        show_velocity=False,
        ylim=OCCUPANCY_YLIM_PCT,
        filename_suffix=FILENAME_SUFFIX,
    )

### Zone Occupancy and Velocity for all Genotypes (% of Bouts)

Plot Zone Occupancy-Velocity plots for all genotypes

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import plot_zone_occupancy_velocity_all


GENOTYPE_SET = "wt_nlf"

plot_zone_occupancy_velocity_all(
    config=config,
    figures_dirname="figures_b/plot7b_zone_occupancy_velocity_pct",
    index_df=df_occupancy_pct,
    zone_names=ZONE_NAMES,
    max_bouts=100,
    n_bins=N_BINS,
    xlabel="% of Bouts",
    title_suffix=" (% of Bouts)",
    show_velocity=False,
    ylim=OCCUPANCY_YLIM_PCT,
    filename_suffix=f"_{GENOTYPE_SET}_{ZONE_SET}_pct",
)

### Zone Occupancy and Velocity per Genotype (Successful Bouts Only, % of Bouts)

Unsuccessful bouts (never reaching target_zone) can't reach overshoot either -- they mechanically contribute occupancy=0 for overshoot-adjacent zones, which dilutes the real signal rather than reflecting it. Restricting to bouts that reached target_zone at least once isolates the genuine behavior (see Plot 7's version of this cell for the exact same rationale).

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import (
    compute_zone_occupancy_velocity_by_percentage,
    plot_zone_occupancy_velocity_all,
)


GENOTYPE_SET = "wt_nlf"

# Restrict to bouts that actually reached the target zone
df_occupancy_pct_successful = compute_zone_occupancy_velocity_by_percentage(
    df_all_csv_wbouts,
    zones=ZONES,
    zone_names=ZONE_NAMES,
    successful_only=True,
)

plot_zone_occupancy_velocity_all(
    config=config,
    figures_dirname="figures_b/plot7b_zone_occupancy_velocity_pct",
    index_df=df_occupancy_pct_successful,
    zone_names=ZONE_NAMES,
    max_bouts=100,
    n_bins=N_BINS,
    xlabel="% of Bouts",
    title_suffix=" (% of Bouts)",
    show_velocity=False,
    ylim=OCCUPANCY_YLIM_PCT,
    filename_suffix=f"_{GENOTYPE_SET}_{ZONE_SET}_pct_successful_only",
)